In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import torch
import torch.nn as nn
from torch.optim import SGD, Adam
from torch.nn import MSELoss, BCEWithLogitsLoss
from source.normal.data import trainLoader
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import ConcatDataset, DataLoader
from source.normal.model import EfficientModel
from source.normal.train import trainModel
from source.normal.loss import CustomLoss
from source.normal.optim import RAdam
from apex import amp

In [3]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/pretrain/train/'
    loader['label_path'] = '../../data/pretrain/train_folds.csv'
    loader['size'] = 330
    loader['fold_idx'] = fold
    loader['weight'] = 1.
    train_1, _ = trainLoader(**loader)
    loader = {}
    loader['image_path'] = '../../data/pretrain/test/'
    loader['label_path'] = '../../data/pretrain/test_folds.csv'
    loader['size'] = 330
    loader['fold_idx'] = fold
    loader['weight'] = 1.
    train_2, _ = trainLoader(**loader)
    loader = {}
    loader['image_path'] = '../../data/train/train/'
    loader['label_path'] = '../../data/train/train_folds.csv'
    loader['size'] = 330
    loader['fold_idx'] = fold
    loader['weight'] = 5.
    train_3, valid = trainLoader(**loader)
    train = ConcatDataset([train_1, train_2, train_3])
    valid = ConcatDataset([valid])
    train = DataLoader(train, batch_size=20, shuffle=True, num_workers=6, drop_last=True)
    valid = DataLoader(valid, batch_size=6, shuffle=True, num_workers=6, drop_last=True)
    model = EfficientModel()
    model = model.to('cuda:0')
    optimizer = RAdam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    schedular = StepLR(optimizer, step_size=5, gamma=0.1)
    model, optimizer = amp.initialize(model, optimizer, opt_level="O2",keep_batchnorm_fp32=True, verbosity=0)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = CustomLoss(weight=0.75, variance=0.2) 
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/combine/model_{}.pt'.format(fold)
    trainer['epochs'] = 10
    trainer['batch'] = 20
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [4]:
train(1)

Train Images: 31611 Valid Images: 3515
Train Images: 48215 Valid Images: 5361
Train Images: 2929 Valid Images: 733
Loaded pretrained weights for efficientnet-b5


100% 82740/82740 [26:59<00:00, 51.10it/s, trn_ls=0.91932, trn_mt=0.61008, val_ls=0.59390, val_mt=0.89478]
/opt/conda/lib/python3.7/site-packages/torch/optim/lr_scheduler.py:73: UserWarning: Seems like `optimizer.step()` has been overridden after learning rate scheduler initialization. Please, make sure to call `optimizer.step()` before `lr_scheduler.step()`. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  "https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate", UserWarning)
100% 82740/82740 [26:55<00:00, 51.22it/s, trn_ls=0.67724, trn_mt=0.75942, val_ls=0.57272, val_mt=0.90528]
100% 82740/82740 [26:58<00:00, 51.13it/s, trn_ls=0.62907, trn_mt=0.79337, val_ls=0.53454, val_mt=0.91801]
100% 82740/82740 [27:01<00:00, 51.01it/s, trn_ls=0.60252, trn_mt=0.81135, val_ls=0.53476, val_mt=0.91410]
100% 82740/82740 [26:57<00:00, 51.16it/s, trn_ls=0.58592, trn_mt=0.82135, val_ls=0.51373, val_mt=0.92216]
100% 82740/82740 [26:57<00:00, 51.1

In [5]:
train(2)

Train Images: 31611 Valid Images: 3515
Train Images: 48216 Valid Images: 5360
Train Images: 2929 Valid Images: 733


  0% 0/82740 [00:00<?, ?it/s]

Loaded pretrained weights for efficientnet-b5


100% 82740/82740 [26:56<00:00, 51.18it/s, trn_ls=0.92140, trn_mt=0.61073, val_ls=0.58410, val_mt=0.88830]
100% 82740/82740 [26:53<00:00, 51.29it/s, trn_ls=0.67491, trn_mt=0.75996, val_ls=0.52722, val_mt=0.91427]
100% 82740/82740 [26:53<00:00, 51.29it/s, trn_ls=0.63097, trn_mt=0.79193, val_ls=0.58919, val_mt=0.90024]
100% 82740/82740 [26:53<00:00, 51.27it/s, trn_ls=0.60263, trn_mt=0.80983, val_ls=0.52038, val_mt=0.91719]
100% 82740/82740 [26:58<00:00, 51.11it/s, trn_ls=0.58547, trn_mt=0.82120, val_ls=0.51144, val_mt=0.92511]
100% 82740/82740 [27:00<00:00, 51.05it/s, trn_ls=0.54672, trn_mt=0.84496, val_ls=0.50521, val_mt=0.92782]
100% 82740/82740 [27:31<00:00, 50.10it/s, trn_ls=0.53315, trn_mt=0.85386, val_ls=0.50016, val_mt=0.93163]
100% 82740/82740 [27:36<00:00, 49.95it/s, trn_ls=0.52784, trn_mt=0.85619, val_ls=0.51733, val_mt=0.92767]
100% 82740/82740 [28:00<00:00, 49.24it/s, trn_ls=0.52155, trn_mt=0.86149, val_ls=0.51002, val_mt=0.92890]
100% 82740/82740 [28:29<00:00, 48.41it/s, trn_

In [6]:
train(5)

Train Images: 31614 Valid Images: 3512
Train Images: 48219 Valid Images: 5357
Train Images: 2931 Valid Images: 731


  0% 0/82760 [00:00<?, ?it/s]

Loaded pretrained weights for efficientnet-b5


100% 82760/82760 [28:21<00:00, 48.63it/s, trn_ls=0.92172, trn_mt=0.60931, val_ls=0.59491, val_mt=0.89311]
100% 82760/82760 [28:25<00:00, 48.53it/s, trn_ls=0.67707, trn_mt=0.75965, val_ls=0.56416, val_mt=0.91214]
100% 82760/82760 [28:23<00:00, 48.57it/s, trn_ls=0.63252, trn_mt=0.79136, val_ls=0.56601, val_mt=0.90039]
100% 82760/82760 [28:24<00:00, 48.55it/s, trn_ls=0.60266, trn_mt=0.80956, val_ls=0.55937, val_mt=0.91159]
100% 82760/82760 [28:23<00:00, 48.58it/s, trn_ls=0.58682, trn_mt=0.82038, val_ls=0.59346, val_mt=0.90925]
100% 82760/82760 [28:24<00:00, 48.57it/s, trn_ls=0.54421, trn_mt=0.84587, val_ls=0.51424, val_mt=0.92318]
100% 82760/82760 [27:30<00:00, 50.14it/s, trn_ls=0.53111, trn_mt=0.85429, val_ls=0.52644, val_mt=0.91851]
100% 82760/82760 [26:57<00:00, 51.18it/s, trn_ls=0.52608, trn_mt=0.85792, val_ls=0.53993, val_mt=0.91533]
100% 82760/82760 [26:54<00:00, 51.27it/s, trn_ls=0.52036, trn_mt=0.86094, val_ls=0.52799, val_mt=0.91841]
100% 82760/82760 [26:51<00:00, 51.37it/s, trn_

In [7]:
train(4)

Train Images: 31614 Valid Images: 3512
Train Images: 48218 Valid Images: 5358
Train Images: 2930 Valid Images: 732


  0% 0/82760 [00:00<?, ?it/s]

Loaded pretrained weights for efficientnet-b5


100% 82760/82760 [26:58<00:00, 51.14it/s, trn_ls=0.91659, trn_mt=0.61447, val_ls=0.59581, val_mt=0.88719]
100% 82760/82760 [26:54<00:00, 51.24it/s, trn_ls=0.67151, trn_mt=0.76271, val_ls=0.59109, val_mt=0.90142]
100% 82760/82760 [26:56<00:00, 51.20it/s, trn_ls=0.63037, trn_mt=0.79287, val_ls=0.54137, val_mt=0.91761]
100% 82760/82760 [27:01<00:00, 51.03it/s, trn_ls=0.60226, trn_mt=0.81047, val_ls=0.52991, val_mt=0.92016]
100% 82760/82760 [26:57<00:00, 51.16it/s, trn_ls=0.58321, trn_mt=0.82408, val_ls=0.54355, val_mt=0.92089]
100% 82760/82760 [27:03<00:00, 50.97it/s, trn_ls=0.54510, trn_mt=0.84574, val_ls=0.51400, val_mt=0.92608]
100% 82760/82760 [27:00<00:00, 51.08it/s, trn_ls=0.53177, trn_mt=0.85466, val_ls=0.51494, val_mt=0.92353]
100% 82760/82760 [27:02<00:00, 51.01it/s, trn_ls=0.52494, trn_mt=0.85805, val_ls=0.53028, val_mt=0.91948]
100% 82760/82760 [26:56<00:00, 51.21it/s, trn_ls=0.51758, trn_mt=0.86278, val_ls=0.52900, val_mt=0.92108]
100% 82760/82760 [27:01<00:00, 51.03it/s, trn_

In [8]:
train(3)

Train Images: 31612 Valid Images: 3514
Train Images: 48217 Valid Images: 5359
Train Images: 2929 Valid Images: 733


  0% 0/82740 [00:00<?, ?it/s]

Loaded pretrained weights for efficientnet-b5


100% 82740/82740 [26:59<00:00, 51.10it/s, trn_ls=0.91654, trn_mt=0.61295, val_ls=0.56778, val_mt=0.90711]
100% 82740/82740 [26:56<00:00, 51.19it/s, trn_ls=0.68003, trn_mt=0.75941, val_ls=0.53939, val_mt=0.91607]
100% 82740/82740 [26:58<00:00, 51.12it/s, trn_ls=0.62873, trn_mt=0.79311, val_ls=0.53264, val_mt=0.90722]
100% 82740/82740 [26:55<00:00, 51.22it/s, trn_ls=0.60517, trn_mt=0.80875, val_ls=0.51398, val_mt=0.91639]
100% 82740/82740 [27:00<00:00, 51.05it/s, trn_ls=0.58614, trn_mt=0.82009, val_ls=0.53532, val_mt=0.91476]
100% 82740/82740 [27:02<00:00, 51.00it/s, trn_ls=0.54347, trn_mt=0.84690, val_ls=0.53024, val_mt=0.91677]
100% 82740/82740 [27:00<00:00, 51.06it/s, trn_ls=0.53326, trn_mt=0.85266, val_ls=0.53537, val_mt=0.91100]
100% 82740/82740 [27:05<00:00, 50.89it/s, trn_ls=0.52593, trn_mt=0.85660, val_ls=0.53596, val_mt=0.91721]
100% 82740/82740 [27:05<00:00, 50.90it/s, trn_ls=0.51848, trn_mt=0.86176, val_ls=0.54950, val_mt=0.91360]
100% 82740/82740 [27:26<00:00, 50.25it/s, trn_